# Intro

This script should be able to process a subrun data and get a star frame and calculate properties of pointing for tha subrun.



# Dependencies and functions

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
import tempfile

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.animation as animation

import pandas as pd
import sep
from PIL import Image, ImageDraw
from scipy import ndimage as ndi

import os
import target_io
from numba import njit
import glob
from datetime import datetime, timezone
import re

# from PIL import Image
import astroalign as aa
aa.MIN_MATCHES_FRACTION = 0.3
aa.NUM_NEAREST_NEIGHBORS = 4  # try 3-4 for sparse fields, minimum is 3

from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
import csv

log_files = "/data/wipac/CTA/targetcdata/run{0}_log.log"
subrun_files = "/data/wipac/CTA/targetcdata/run{0}_subrun{1}_r1.tio"
catalog_path = "/home/oriss/star_detection_pSCT/hyg_v42.csv"
telescope_config = "/home/oriss/star_detection_pSCT/psct_config.txt"

make_nans = False

In [ ]:
pxs_p_quad = 16
quads_p_module = 4

sources_of_interest = [
    {'name': 'Mrk 421', 'ra': 166.11380833, 'dec': 38.208833, 'mag': 12.9},
    # {'name': '51 UMa', 'ra': 166.13016, 'dec': 38.241365, 'mag': 6.28},
    ]

others_bounds = [[0, np.exp(4)], [0, np.exp(4)]] # "Others" event box bounds

fpms_t = ['7-21', '7-22', '7-23', '7-24', '7-15', '7-16', '7-17', '7-18', '7-19', '7-10', '7-11', '7-12', '7-13', '7-14', '7-5', '7-6', '7-7', '7-8', '7-9', '7-0', '7-1', '7-20', '7-2', '7-3', '7-4']


ups_mods = { # Note this list modules that do not exist. This is because I just care about identifying positions which would flip upsidown if they were to be placed. 
    "0": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "2": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "3": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "5": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "6": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "8": [0, 2, 4, 5, 7, 9, 10, 12, 14, 15, 17, 19, 20, 22, 24],
    "1": [1, 3, 6, 8, 11, 13, 16, 18, 21, 23],
    "4": [1, 3, 6, 8, 11, 13, 16, 18, 21, 23],
    "7": [1, 3, 6, 8, 11, 13, 16, 18, 21, 23],
}

fpms_exist = {
    "0": [8, 9, 12, 13, 14, 16, 17, 18, 19, 21, 22, 23, 24],
    "2": [5, 6, 10, 11, 12, 15, 16, 17, 18, 20, 21, 22, 23],
    "3": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
    "5": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
    "6": [1, 2, 3, 4, 6, 7, 8, 9, 12, 13, 14, 18, 19],
    "8": [0, 1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 15, 16],
    "1": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
    "4": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
    "7": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
}

def load_config(filepath):
    result = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                key, value = line.split(':', 1)
                result[key.strip()] = value.strip()
    return result

@njit
def getarrs(wfs_a):
    wfs_me = [np.zeros((1600), dtype=np.float64) for _ in wfs_a]
    wfs_sq = [np.zeros((1600), dtype=np.float64) for _ in wfs_a]
    for ev, wf in enumerate(wfs_a):
        for ch in range(wf.shape[0]):
            wfs_me[ev][ch] = np.mean(wf[ch])
            wfs_sq[ev][ch] = np.mean(wf[ch]**2)
    return wfs_me, wfs_sq

@njit
def get_gbmean(wfs_m, wfs_sm):
    N = len(wfs_m)
    gbmean = np.zeros(wfs_m[0].shape)
    for wf in wfs_m:
        gbmean = gbmean + wf
    return gbmean/N

@njit
def get_int_chargemeanstd_ev(wf, int_win=2):
    n_channels, n_samples = wf.shape
    int_charge = np.zeros((n_channels))
    wf[4*64 + 14, :] = np.nan
    wf[5*64:6*64, :] = np.nan
    wf[13*64:14*64, :] = np.nan
    wf[21*64:, :] = np.nan
    for ch in range(n_channels):
        t_max = np.argmax(wf[ch])
        if t_max > n_samples - int_win - 1:
            t_max = n_samples - int_win - 1
        elif t_max < int_win:
            t_max = int_win
        int_charge[ch] = wf[ch, t_max-int_win:t_max+int_win+1].sum()
    mean_c = np.nanmean(int_charge)
    std_c = np.nanstd(int_charge)
    
    return mean_c, std_c


def read_wfs_metrics(calfile, save=False, reader = None):
    if reader == None:
        reader = target_io.WaveformArrayReader(calfile, silent=True)
    wfs_me = []
    wfs_sq = []
    times = []
    wfs_all = []
    for ev in range(reader.fNEvents):
        wfs = np.zeros((reader.fNPixels, reader.fNSamples), dtype=np.float32)

        reader.GetR1Event(ev, wfs)
        mean_c, std_c = get_int_chargemeanstd_ev(wfs, 4)
        if mean_c < others_bounds[0][1] and std_c < others_bounds[1][1]:
            wfs_all.append(wfs)
            times.append(reader.fTACK_time)
    aa, bb = getarrs(wfs_all)
    wfs_me = wfs_me + aa
    wfs_sq = wfs_sq + bb
        
    # all_wfs = np.array(all_wfs)
    
    for wf_m, wf_s in zip(wfs_me, wfs_sq):
        wf_m[4*64 + 14] = 0.0
        # for l in range(64):
        wf_m[5*64:6*64] = 0.0
        wf_m[13*64:14*64] = 0.0
        wf_m[21*64:] = 0.0

        wf_s[4*64 + 14] = 0.0
        # for l in range(64):
        wf_s[5*64:6*64] = 0.0
        wf_s[13*64:14*64] = 0.0
        wf_s[21*64:] = 0.0
    return np.array(wfs_me), np.array(wfs_sq), np.array(times)



def extract_timestamp(file_path, preceding_string, add_value = 0.0):
    with open(file_path, 'r') as f:
        content = f.read()
    
    pattern = re.escape(preceding_string) + r'([\d]+\.[\d]+)'
    match = re.search(pattern, content)
    
    if match:
        return float(match.group(1))+add_value
    else:
        raise ValueError(f"No number found after '{preceding_string}'")

def get_time_base(run_num):
    try:
        base_timestamp = extract_timestamp(log_files.format(run_num), "- INFO     - Clock origin timestamp: ")
    except:
        try:
            base_timestamp = extract_timestamp(log_files.format(run_num), "Sub-Run 0 Start Time: * / ")
        except:
            base_timestamp = None
    return base_timestamp

def fetch_pointing(run, sr, time):
    """
    Figure a function that takes the pointing 
    """
    return None, None # ra, dec in degrees

def convert_to_utc_str(time):
    utc_dt = datetime.fromtimestamp(time, tz=timezone.utc)
    return utc_dt.strftime("%Y-%m-%dT%H:%M:%S")


def CTC_lab_im_map():
    """
    Calculates the grid index, which is used to go from (64) pixel data to (8, 8) lab image. Includes TARGET to SiPM mapping.

    :return: grid index
    :rtype: numba.typed.List
    """

    ch_nums = np.array([[23,22,19,18, 4, 5, 0, 1],
                        [21,20,17,16, 6, 7, 2, 3],
                        [30,31,27,26,12,13, 8, 9],
                        [29,28,25,24,14,15,10,11],
                        [55,54,51,50,36,37,32,33],
                        [53,52,49,48,38,39,34,35],
                        [62,63,59,58,44,45,40,41],
                        [61,60,57,56,46,47,42,43]])
    ch_nums_1D = ch_nums.reshape(-1)
    ch_to_pos = dict(zip(ch_nums_1D, np.arange(64)))

    total_cells = 64
    indices = np.arange(total_cells).reshape(-1, int(np.sqrt(total_cells)))
    grid_ind = list()

    i, j = 0, 0
    ch_map = dict()
    ch_map = ch_to_pos
    pix_ind = np.array(indices[(8*i):8*(i+1), (8*j):8*(j+1)]).reshape(-1)
    for asic in range(4):
        for ch in range(16):
            grid_ind.append(int(pix_ind[ch_map[asic * 16 + ch]]))
            
    return grid_ind

grid_ind0 = CTC_lab_im_map()

def make_grid_module(vals, a = None, x0 = 0, y0 = 0, upsideup = True, grid_ind = grid_ind0):
# grid_ind = CTC_lab_im_map()
    try:
        bb = a[0][0]
    except:
        a = np.empty(shape = (8+x0, 8+y0))
        if make_nans: a[:] = np.nan
        else: a[:] = 0.0
    bb = a.shape
    if bb[0] < x0 + 8:
        b = a
        a = np.empty(shape = (x0+8, a.shape[1]))
        a[:b.shape[0]][:b.shape[1]] = b
        bb = a.shape
    if bb[1] < y0 + 8:
        b = a
        a = np.empty(shape = (a.shape[0], y0+8))
        a[:b.shape[0]][:b.shape[1]] = b

    # vals = np.array(vals)
    # if len(vals) < 64:
    #     vals = np.pad(vals.astype(float), (0, 64 - len(vals)), constant_values=np.nan)
    if all(x == 0.0 for x in vals) and make_nans:
        vals[:] = np.nan
    for i, val in enumerate(vals):
        xy = grid_ind[i]
        y, x = xy // 8, xy % 8
        # x, y = xy % 8, xy // 8
        if upsideup:
            y =  7 - y
            x = 7 - x
        a[x0+x, y0+y] = val
    return np.array(a)
    # return np.array(list(reversed(list(list(zip(*a))))))

def make_grid_sector(valss, fpms,  a = None, x0 = 0, y0 = 0, sector = None, grid_ind = grid_ind0):
    try:
        bb = a[0][0]
    except:
        a = np.empty(shape = (40+x0, 40+y0))
        a[:] = np.nan
    bb = a.shape
    if bb[0] < x0 + 40:
        b = a
        a = np.empty(shape = (x0+40, a.shape[1]))
        a[:b.shape[0]][:b.shape[1]] = b
    bb = a.shape
    if bb[1] < y0 + 40:
        b = a
        a = np.empty(shape = (a.shape[0], y0+40))
        a[:b.shape[0]][:b.shape[1]] = b
    valss = [vals for vals in valss]
    if not make_nans:
        if sector == None and len(fpms) > 0:
            sector, _ = fpms[0].split("-")
        elif sector == None:
            sector = "4" # this will be a blank picture I think
        for fp in fpms_exist[sector]:
            if f"{sector}-{fp}" not in fpms:
                fpms.append(f"{sector}-{fp}")
                valss.append([0.0 for _ in range(pxs_p_quad*quads_p_module)])

    for i, vals in zip(fpms, valss):
        sector, i = i.split("-")
        i = int(i)
        x00 = (i*8 % 40)
        y00 = (8*(4 - (i*8 // 40)))
        if i in ups_mods[str(sector)]:
            upsideup = True
        else:
            upsideup = False
        a = make_grid_module(vals, a, x0+x00, y0+y00, upsideup, grid_ind = grid_ind)
    return np.array(a)

def make_grid_camera(valss, fpms,  a = None, grid_ind = grid_ind0):
    a = np.empty(shape = (120, 120))
    a[:] = np.nan

    fpmss = {"0": [],
             "1": [],
             "2": [],
             "3": [],
             "4": [],
             "5": [],
             "6": [],
             "7": [],
             "8": [],
    }

    valsss = {"0": [],
             "1": [],
             "2": [],
             "3": [],
             "4": [],
             "5": [],
             "6": [],
             "7": [],
             "8": [],
    }

    for val, fpm in zip(valss, fpms):
        sector, _ = fpm.split("-")
        fpmss[sector].append(fpm)
        valsss[sector].append(val)

    for i in range(9):
        sector = i
        i = f"{i}"
        fpms = fpmss[i]
        valss = valsss[i]
        # sector, _ = fpms[0].split("-")
        # sector = int(sector)
        x0 = (sector*40) % 120
        y0 = 80 - 40*(sector // 3)
        a = make_grid_sector(valss, fpms, a, x0, y0, grid_ind = grid_ind, sector = i)
    # return np.array(a)
    return np.array(list(reversed(list(list(zip(*a))))))


def rotate(a, nturns = 0, clockwise = True):
    if nturns > 0:
        for _ in range(nturns):
            if clockwise:
                a = list(reversed(list(list(zip(*a)))))
            else:
                a = list(reversed(list(list(zip(*a))[::-1])))
    return np.array(a)


def get_visible_stars(ra_center, dec_center, fov_deg, xlen, ylen,
                       time_utc, lat, lon, elevation=0.0, catalog=None):
    """
    Returns stars within a field of view as seen through an Alt/Az-mounted
    telescope: x increases toward azimuth-right (observer's right when
    facing the target), y increases toward zenith. Origin (0,0) is
    bottom-left. Field rotation over time falls out naturally—no separate
    rotation step needed.

    Written with help of Claude
    """
    if catalog is None:
        raise Exception(f"Provide a star catalog!")

    location = EarthLocation(lat=lat * u.deg, lon=lon * u.deg, height=elevation * u.m)
    obstime = Time(time_utc)                       # fresh each call
    altaz_frame = AltAz(obstime=obstime, location=location)  # fresh each call

    center = SkyCoord(ra=ra_center * u.deg, dec=dec_center * u.deg, frame="icrs")
    names = [s["name"] for s in catalog]
    magnitudes = [s["mag"] for s in catalog]
    coords = SkyCoord(ra=[s["ra"] for s in catalog] * u.deg,
                       dec=[s["dec"] for s in catalog] * u.deg,
                       frame="icrs")

    half_fov = fov_deg / 2.0
    mask = coords.separation(center).deg <= half_fov
    if not np.any(mask):
        return []
    names = [n for n, m in zip(names, mask) if m]
    coords = coords[mask]

    # Transform to Alt/Az fresh, using this call's obstime/location
    altaz = coords.transform_to(altaz_frame)
    center_altaz = center.transform_to(altaz_frame)

    alt0, az0 = center_altaz.alt.rad, center_altaz.az.rad
    alt, az = altaz.alt.rad, altaz.az.rad

    cos_c = np.cos(alt0) * np.cos(alt) * np.cos(az - az0) + np.sin(alt0) * np.sin(alt)
    angg = np.arccos((np.sin(alt)-np.sin(alt0)*cos_c)/(np.cos(alt0)*(np.sin(np.arccos(cos_c)))))

    xi = np.cos(alt) * np.sin(az - az0) / cos_c
    eta = (np.cos(alt0) * np.sin(alt) - np.sin(alt0) * np.cos(alt) * np.cos(az - az0)) / cos_c
    xi_deg, eta_deg = np.rad2deg(xi), np.rad2deg(eta)


    px = (xi_deg + half_fov) / fov_deg * xlen
    py = (eta_deg + half_fov) / fov_deg * ylen   # origin bottom-left, y up toward zenith

    return [
        {
            "name": names[i],
            "ra": coords.ra.deg[i],
            "dec": coords.dec.deg[i],
            "alt": altaz.alt.deg[i],
            "az": altaz.az.deg[i],
            "x": float(px[i]),
            "y": float(py[i]),
            "mag": float(magnitudes[i])
        }
        for i in range(len(names))
    ]


def load_hyg_catalog(path, mag_limit=6.0, named_only=False):
    """
    Loads stars from a HYG database CSV (e.g. hygdata_v3.csv).
    Download from: https://github.com/astronexus/HYG-Database

    path        : str   - path to hygdata_v3.csv
    mag_limit   : float - only include stars brighter than this apparent magnitude
    named_only  : bool  - if True, skip stars without a proper name
                          (most of the ~119,000 rows only have catalog IDs)

    Returns list of {"name", "ra", "dec", "mag"} dicts, ra/dec in degrees.

    Written with help of Claude
    """
    catalog = []
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                mag = float(row["mag"])
            except (ValueError, KeyError):
                continue
            if mag > mag_limit:
                continue

            proper = (row.get("proper") or "").strip()
            if named_only and not proper:
                # print("A")
                continue

            try:
                ra_deg = float(row["ra"]) * 15.0   # HYG stores RA in hours
                dec_deg = float(row["dec"])
            except (ValueError, KeyError):
                continue

            if proper:
                # print("B")
                name = proper
            else:
                # fall back to Bayer/Flamsteed designation, then HD/HIP, then row id
                bayer = (row.get("bf") or "").strip()
                hd = (row.get("hd") or "").strip()
                hip = (row.get("hip") or "").strip()
                name = bayer or (f"HD {hd}" if hd else None) or (f"HIP {hip}" if hip else f"HYG {row.get('id')}")

            catalog.append({"name": name, "ra": ra_deg, "dec": dec_deg, "mag": mag})

    return catalog


def clean_image(a0, p1 = 8, p2 = 4):
    
    # a_bkg_analysis = a0.copy()

    # a_bkg_analysis[a_bkg_analysis == 0.0] = np.nan
    # bkg_image, bkg_rms, bkg = estimate_background(a_bkg_analysis, bw = p1, bh = p1, fw = p2, fh = p2)
    # bkg_image[a0 == 0.0] = 0.0

    # a = a0 - bkg_image

    bkg_base = np.load('./background.npy') # For now
    a = a0 - bkg_base*6.5*np.nanmean(a0)

    return a

def get_star_parameters_from_subrun(a0, time_str, ra_center, dec_center, psct_config, catalog, sources = None):

    if sources != None:
        sources_list = sources
    else:
        sources_list = []
    a = clean_image(a0) # This cleans the 2D image. Need to improve
    a = a[:, ::-1] # Mirrors it horizontally (from camera view to sky view)
    a = rotate(a, 2, True) # Rotates 180 degrees - now the image matches the sky, since a 180 rotation is the effect of the two mirrors

    # Extracting objects in the image:
    a_pro = a.copy()
    a_pro[a_pro == 0.0] = np.nan # Pixels that did not fetch data are usually at 0.0 -> go to nan
    a_mask = a_pro.copy() # For a boolean mask
    a_mask[a_pro == np.nan] = False
    a_mask[a_pro != np.nan] = True
    bkg = sep.Background(a, mask = a_mask) 
    objects = sep.extract(a - bkg, 1.0, minarea=2, filter_kernel=None)
    x_detected = np.array([obj[7] for obj in objects])
    y_detected = np.array([obj[8] for obj in objects])

    # Listing possible stars in the field
    stars = get_visible_stars(
            ra_center=ra_center, dec_center=dec_center, fov_deg=float(psct_config['fov_deg']), # Function
            xlen=120, ylen=120,
            time_utc=time_str,
            lat=float(psct_config['lat']), lon=float(psct_config['lon']), elevation=float(psct_config['elevation']),
            catalog=catalog
        )
    
    sources_pred = get_visible_stars(
            ra_center=ra_center, dec_center=dec_center, fov_deg=float(psct_config['fov_deg']), # Function
            xlen=120, ylen=120,
            time_utc=time_str,
            lat=float(psct_config['lat']), lon=float(psct_config['lon']), elevation=float(psct_config['elevation']),
            catalog=sources_list
        )
    x_predicted = np.array([s['x'] for s in stars])
    y_predicted = np.array([s['y'] for s in stars])

    x_src_predicted = np.array([s['x'] for s in sources_pred])
    y_src_predicted = np.array([s['y'] for s in sources_pred])

    nearby_predicted = [(x-60, y-60) for x, y in zip(x_predicted, y_predicted) if np.min((x - x_detected)**2+(y - y_detected)**2) < 64]

    found_turples = [(x-60, y-60) for x, y in zip(x_detected, y_detected)]

    sources_turple = [(x-60, y-60) for x, y in zip(x_src_predicted, y_src_predicted)]

    aa.MIN_MATCHES_FRACTION = 0.1
    k = max(len(nearby_predicted), len(found_turples))
    aa.NUM_NEAREST_NEIGHBORS = k+1
    try:
        transf, (source_list, target_list) = aa.find_transform(nearby_predicted, found_turples)
        dst_calc = aa.matrix_transform(sources_turple, transf.params)

        for ind, x, y in zip(range(len(sources_pred)), [ob[0] for ob in dst_calc], [ob[1] for ob in dst_calc]):
            sources_pred[ind]['x'] = x
            sources_pred[ind]['y'] = y

        dst_calc = aa.matrix_transform([(x-60, y-60) for x, y in zip(x_predicted, y_predicted)], transf.params)
        for ind, x, y in zip(range(len(stars)), [ob[0] for ob in dst_calc], [ob[1] for ob in dst_calc]):
            stars[ind]['x'] = x
            stars[ind]['y'] = y


        deltas, rot_ang = transf.translation, transf.rotation
        delta_x, delta_y = deltas
        return a-bkg, objects, x_detected, y_detected, stars, x_predicted, y_predicted, sources_pred, delta_x, delta_y, rot_ang, transf
    except:
        print(f'Finding a match failed!')
        return a-bkg, objects, x_detected, y_detected, stars, x_predicted, y_predicted, sources_pred, None, None, None, None

# Collecting subrun

In [ ]:
run = 400215 # Will be an argument for a python calleable script
sr = 6 # Will be an argument for a python calleable script

psct_config = load_config(telescope_config) # Gets information about the telescope, like longitude, latitude, elevation, FoV
catalog = load_hyg_catalog("hyg_v42.csv", mag_limit=10, named_only=False) # Loads star catalog

reader_sr0 = target_io.WaveformArrayReader(subrun_files.format(run, 0)) # Load reader object for the first subrun
reader_sr0.GetR1Event(0, np.zeros((reader_sr0.fNPixels, reader_sr0.fNSamples), dtype = np.float32)) # Making the 0th event actively present in the reader object
time_0 = reader_sr0.fTACK_time # fetching the time for this first event
del reader_sr0 # Deleting object
reader = target_io.WaveformArrayReader(subrun_files.format(run, sr)) # Load reader object

wfs_me, wfs_sq, times = read_wfs_metrics(None, reader = reader) # Calculate the waveforms means and mean of the squares, per pixel per event (this filters events with strong charges out)

# time_base = get_time_base(run) # This will take the time the run starts. The 400215 log fails to have a clear flag so I manually put the start of subrun 0 bellow
time_base = 1776574920.7772014
time_abs = time_base + (np.mean(times)-time_0)/1E9
time_str = convert_to_utc_str(time_abs) # This gets the time in the format we want. I am estimating the time of the frame based on the mean of the times of the events involved
print(time_str) # Frame time in UTC of run

ra_center, dec_center = fetch_pointing(run, sr, time_abs)
ra_center, dec_center = 168.24942499999997, 36.13756194444444 # For 400215

In [ ]:
frames_ev = (wfs_sq - wfs_me**2).reshape(len(wfs_sq), 1600 // 64, 64) # Gets the image frame per event

frame_arr = get_gbmean(frames_ev, None).reshape(1600 // 64, 64) # Gets the mean of that frame and reshape the object. This is not yet a 2D image

a0 = make_grid_camera(frame_arr, fpms=fpms_t) # This makes the 2D image


a, objects, x_detected, y_detected, stars, x_predicted, y_predicted, gamma_sources, delta_x, delta_y, rot_ang, transf = get_star_parameters_from_subrun(a0, time_str, ra_center, dec_center, psct_config, catalog, sources_of_interest) # This gets all the information relating to stars: image, detected objects, x_values of detected objects, y_values of detected objects



Run the next block to get the image

In [ ]:

# Plot camera frame
fig, ax = plt.subplots(squeeze = True, layout = 'tight')

im = ax.imshow(a, interpolation=None, origin = 'lower', vmin = 0, extent = [-60.5, 59.5, -60.5, 59.5])

# Plotting markers for detected stars
for x, y, flu in zip(x_detected, y_detected, [obj[21] for obj in objects]):

    ax.plot(x-60, y-60, marker = "x", color = 'red', markersize = 7*np.sqrt(flu/75))


# Plotting markers for possible stars in the field
for x, y, flu, name in zip([s['x'] for s in stars], [s['y'] for s in stars], [10**(-s['mag']/2.5) for s in stars], [s['name'] for s in stars]):
    if np.min((x - x_detected+60)**2+(y - y_detected+60)**2) < 64:
        ax.plot(x, y, marker = "*", color = 'yellow')#, markersize = 7*np.sqrt(flu/75))
    else:
        ax.plot(x, y, marker = "*", color = 'yellow', alpha = 0.3)
        # ax.text(x-60, y-60, f'{name}', fontsize = 5)

for x, y, name in zip([ob['x'] for ob in gamma_sources], [ob['y'] for ob in gamma_sources], [ob['name'] for ob in gamma_sources]):

    ax.plot(x, y, marker = "+", color = "#01FFEE", markersize = 15)
    ax.text(x, y, f'{name}', fontsize = 15)


cbar = plt.colorbar(im, ax=ax,)
cbar.set_label(r'$\left< \text{Var} \right> \text{ } [\text{ADC}^2 \text{ns}^2]$')
ax.set_title(f"Run {run}, Subrun {sr}\n {time_str} UTC\n" + r"$\Delta x \equal$" +f"{delta_x:.2f}, " + r"$\Delta y \equal$" +f"{delta_y:.2f}, " + r"$\Delta \theta \equal$" +f"{rot_ang:.4f}", fontsize = 'x-small')
plt.show(False)
ax.set_xlim(-20.5, 19.5)
ax.set_ylim(-20.5, -60.5)
plt.show()
for gs in gamma_sources:
    print(f"{gs['name']}: \n x: {gs['x']} \n y: {gs['y']}")


# Analysing the full run

In [ ]:
run = 400215
wfs_me = np.load(f'/data/user/oriss/data_stars/wfs_mean_run{run}_B.npy')
wfs_sq = np.load(f'/data/user/oriss/data_stars/wfs_mean_sq_run{run}_B.npy')
times = np.load(f'/data/user/oriss/data_stars/wfs_times_run{run}_B.npy')



In [ ]:
ra_center, dec_center = 168.24942499999997, 36.13756194444444 # For 400215
psct_config = load_config(telescope_config) # Gets information about the telescope, like longitude, latitude, elevation, FoV
catalog = load_hyg_catalog("hyg_v42.csv", mag_limit=10, named_only=False) # Loads star catalog


frames = []
time_strs = []
arg = 0
arg_l = 0
P = 60*1E9
# base_timestamp = get_time_base(run)
base_timestamp = 1776574920.7772014
timess = [int(t) for t in times]
abs_times = []
print(timess[0])
for ind, t in enumerate(timess):
    if (t - timess[arg_l]) > P:
        arg = arg_l
        arg_l = ind

        frames.append(get_gbmean([(wfs_sq_on - wfs_me_on**2) for wfs_sq_on, wfs_me_on in zip(wfs_sq[arg:arg_l], wfs_me[arg:arg_l])], None).reshape(1600 // 64, 64))

        delta_t = (np.mean(timess[arg:arg_l+1]) - timess[0])/(1E9)
        timestamp_ev = convert_to_utc_str(base_timestamp + delta_t)
        abs_times.append(base_timestamp + delta_t)
        time_strs.append(timestamp_ev)

frames_arr = np.array(frames)


frames_med = frames_arr.copy()
frames_med.sort(axis=0)
frames_med = np.median(frames_med, axis = 0)

frames_med = frames_med/np.mean(frames_med)

wfs_plot = [np.array([fra - np.mean(fra)*frames_med for ind, fra in enumerate(frames_arr)])]
bkg_base = np.load('./background.npy')
list_frames = []
for wf in wfs_plot[0]:
    a0 = make_grid_camera(wf, fpms=fpms_t)
    a = a0 + bkg_base*6.5*np.nanmean(a0)
    list_frames.append(np.array(a))

In [ ]:
dict_data = {
    'name': [],
    'x': [],
    'y': [],
    'time_utc': [],
    'time_abs': []
}

offset_data = {
    'x_c': [],
    'y_c': [],
    'time_utc': [],
    'time_abs': []
}

for a0, time in zip(list_frames, abs_times):
    time_str = convert_to_utc_str(time)
    a, objects, x_detected, y_detected, stars, x_predicted, y_predicted, dst_calc, delta_x, delta_y, rot_ang, transf = get_star_parameters_from_subrun(a0, time_str, ra_center, dec_center, psct_config, catalog, sources_of_interest)
    if delta_x != None:
        # Plot camera frame
        fig, ax = plt.subplots(squeeze = True, layout = 'tight', figsize = (5,5))

        im = ax.imshow(a, interpolation=None, origin = 'lower', vmin = 0, vmax = 5, extent = [-60.5, 59.5, -60.5, 59.5])

        # Plotting markers for detected stars
        for x, y, flu in zip(x_detected, y_detected, [obj[21] for obj in objects]):

            ax.plot(x-60, y-60, marker = "x", color = 'red', markersize = 7*np.sqrt(flu/75))


        # Plotting markers for possible stars in the field
        for x, y, flu, name in zip([s['x'] for s in stars], [s['y'] for s in stars], [10**(-s['mag']/2.5) for s in stars], [s['name'] for s in stars]):
            if np.min((x - x_detected+60)**2+(y - y_detected+60)**2) < 15:
                # ax.plot(x, y, marker = "*", color = 'yellow', alpha = 0.6)#, markersize = 7*np.sqrt(flu/75))
                if name in ['49    UMa']:
                    ax.text(x, y, f'{name}', fontsize = 10)
            # else:
                # ax.plot(x, y, marker = "*", color = 'yellow', alpha = 0.1)


        for x, y, name in zip([ob['x'] for ob in dst_calc], [ob['y'] for ob in dst_calc], [ob['name'] for ob in dst_calc]):

            ax.plot(x, y, marker = "+", color = "#01FFEE", markersize = 10)
            ax.text(x, y, f'{name}', fontsize = 15)

        dict_data['name'].append('Mrk 421')
        dict_data['x'].append(dst_calc[0]['x'])
        dict_data['y'].append(dst_calc[0]['y'])
        dict_data['time_abs'].append(time)
        dict_data['time_utc'].append(time_str)

        offset_data['x_c'].append(delta_x)
        offset_data['y_c'].append(delta_y)
        offset_data['time_abs'].append(time)
        offset_data['time_utc'].append(time_str)


        
        cbar = plt.colorbar(im, ax=ax,)
        cbar.set_label(r'$\left< \text{Var} \right> \text{ } [\text{ADC}^2 \text{ns}^2]$')
        ax.set_title(f"Run {run}\n {time_str} UTC\n" + r"$\Delta x \equal$" +f"{delta_x:.2f}, " + r"$\Delta y \equal$" +f"{delta_y:.2f}, " + r"$\Delta \theta \equal$" +f"{rot_ang:.4f}", fontsize = 'x-small')
        ax.set_xlim(-20.5, 19.5)
        ax.set_ylim(-19.5, -60)
        fig.set_dpi(800)
        fig.savefig(f"/data/user/oriss/plots/run400215_star_matched/run400215_{time_str.replace(':','-')}.jpeg")
        plt.show()

pd.DataFrame.from_dict(dict_data).to_csv('/data/user/oriss/plots/run400215_star_matched/run400215_positions_mrk421.csv')
pd.DataFrame.from_dict(offset_data).to_csv('/data/user/oriss/plots/run400215_star_matched/run400215_center_offsets.csv')


# Movie of run (In development)

In [ ]:
def animate_frames(arrays, other_args = None, interval=250, save = None, time_arr = None):
    """
    Parameters
    ----------
    arrays   : list of 2D numpy arrays, one per frame
    plot_fn  : your function that takes a 2D array and returns (fig, ax)
    interval : milliseconds between frames
    
    Returns
    -------
    fig, anim
    """
    interval = int(interval)
    # other_args[7] = None
    # other_args[8] = False
    fig, ax = camera_image(arrays[0], 3, False, True)
    time_text = ax.text(
        0.45, -0.05,           # x, y in axes coordinates (top-left area)
        '',                    # starts empty
        transform=ax.transAxes,
        # fontsize=time_legend_font,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.0)
    )
    im = ax.images[0]  # grab the AxesImage created by imshow

    def update(frame_idx):
        im.set_data(arrays[frame_idx])
        current_time = frame_idx # in ns
        if time_arr == None:
            time_text.set_text(f't = {current_time} ns')
        else:
            time_text.set_text(f't = {time_arr[frame_idx]}')
        
        return (im,)

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=len(arrays),
        interval=interval,
        blit=True,
    )

    if save != None:
        # To save the animation using Pillow as a gif
        writer = animation.PillowWriter(fps=int(1000/interval),
                                        metadata=dict(artist='Oris Salviano Neto'),
                                        bitrate=1800)
        anim.save(save, writer=writer)
    return fig, anim